# PCA Process Monitor — Processo Batch

Dimostrazione completa su dataset sintetico di polimerizzazione batch.

**Flusso:**
1. Setup e caricamento dati batch
2. Overview e statistica descrittiva
3. Batch-wise unfolding
4. Costruzione modello NIPALS — selezione PC
5. Score plot (un punto = un batch), loadings nel tempo
6. Diagnostica post-mortem sui batch di test
7. Diagnostica on-line (simulazione batch in corso)
8. AI interpreter (opzionale)


## 0 — Setup

In [ ]:
import os, sys
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'pca-process-monitor'
USER = 'MarDan93'

if not os.path.exists(f'/content/{REPO}'):
    os.system(f'git clone https://{GITHUB_TOKEN}@github.com/{USER}/{REPO}.git')

os.chdir(f'/content/{REPO}')
sys.path.insert(0, f'/content/{REPO}')
print(f'Working dir: {os.getcwd()}')

In [ ]:
!pip install -q numpy pandas matplotlib scipy plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from pca_monitor.nipals        import NIPALS
from pca_monitor.preprocessing import (
    detect_process_type, missing_summary, handle_missing,
    unfold_batch, dataframe_info, get_numeric_columns
)
from pca_monitor.diagnostics   import (
    BatchPostMortem, BatchOnlineMonitor,
    find_anomalies, compute_cal_contribution_limits
)
from pca_monitor import plots

print('✓ Package caricato correttamente')

## 1 — Caricamento dati batch

In [ ]:
!python data/generate_datasets.py

In [ ]:
df_cal  = pd.read_csv('data/batch_calibration.csv')
df_test = pd.read_csv('data/batch_test.csv')

print('=== CALIBRAZIONE ===')
print(f'Shape: {df_cal.shape}')
print(f'Batch unici   : {df_cal["batch_id"].nunique()}')
print(f'Istanti/batch : {df_cal["time"].nunique()}')
print(df_cal.head())
print('\n=== TEST ===')
print(f'Batch unici   : {df_test["batch_id"].nunique()}')
print(df_test.head())

## 2 — Overview e statistica descrittiva

In [ ]:
# Rilevamento tipo processo
detection = detect_process_type(df_cal)
print(f"Tipo rilevato : {detection['type']}")
print(f"Confidenza    : {detection['confidence']}")
print(f"Motivazione   : {detection['reason']}")
print(f"Batch col     : {detection['batch_col']}")
print(f"Time col      : {detection['time_col']}")

In [ ]:
# Variabili di processo
VAR_COLS = get_numeric_columns(df_cal, exclude=['time', 'split'])
print(f'Variabili di processo: {VAR_COLS}')

BATCH_COL = 'batch_id'
TIME_COL  = 'time'
N_TIME    = df_cal[TIME_COL].nunique()
N_VARS    = len(VAR_COLS)
print(f'Istanti temporali    : {N_TIME}')
print(f'Numero variabili     : {N_VARS}')

In [ ]:
# Distribuzione variabili (tutti i batch sovrapposti)
fig = plots.plot_variable_distributions(
    df_cal, VAR_COLS, kind='timeseries', ncols=3
)
plt.suptitle('Traiettorie variabili — tutti i batch calibrazione',
             fontsize=13, fontweight='bold', y=1.01)
plt.show()

In [ ]:
# Statistica descrittiva
df_cal[VAR_COLS].describe().round(3)

## 3 — Batch-wise unfolding

In [ ]:
# Unfolding calibrazione
X_cal_unf, batch_ids_cal, col_names = unfold_batch(
    df        = df_cal,
    batch_col = BATCH_COL,
    time_col  = TIME_COL,
    var_cols  = VAR_COLS
)

print(f'Matrice unfolded calibrazione: {X_cal_unf.shape}')
print(f'  Righe  = batch ({len(batch_ids_cal)})')
print(f'  Colonne = variabili × istanti ({N_VARS} × {N_TIME} = {N_VARS*N_TIME})')
print(f'\nPrime colonne: {col_names[:6]}')
print(f'Ultime colonne: {col_names[-3:]}')

In [ ]:
# Unfolding test
X_test_unf, batch_ids_test, _ = unfold_batch(
    df        = df_test,
    batch_col = BATCH_COL,
    time_col  = TIME_COL,
    var_cols  = VAR_COLS
)

print(f'Matrice unfolded test: {X_test_unf.shape}')
print(f'Batch test: {batch_ids_test}')

## 4 — Costruzione modello NIPALS e selezione PC

In [ ]:
# Fit modello su dati unfolded
MAX_PC = 8
model  = NIPALS(n_components=MAX_PC, scale='auto', tol=1e-6, max_iter=500)
model.fit(X_cal_unf)
model.summary()

In [ ]:
# Scree plot
fig = plots.plot_scree(
    eigenvalues   = model.eigenvalues,
    explained_var = model.explained_variance_ratio_
)
plt.show()

In [ ]:
# RMSECV
print('Calcolo RMSECV in corso...')
rmsecv_vals, optimal_nc = model.rmsecv(
    X_cal_unf, max_components=MAX_PC, cv_folds=5
)
print(f'Numero ottimale PC (RMSECV): {optimal_nc}')

fig = plots.plot_rmsecv(rmsecv_vals, optimal_nc)
plt.show()

N_PC = optimal_nc
print(f'PC usate per la diagnostica: {N_PC}')

## 5 — Score plot e loadings nel tempo

In [ ]:
# Score plot calibrazione — ogni punto è un batch
PC_X, PC_Y = 1, 2

fig = plots.plot_scores(
    T      = model.T,
    pc_x   = PC_X,
    pc_y   = PC_Y,
    labels = batch_ids_cal,
    alpha  = 0.05,
    title  = 'Score plot batch — calibrazione (un punto = un batch)'
)
plt.show()

In [ ]:
# Loading nel tempo per variabile selezionata
# Mostra il contributo di ogni variabile alla PC in ogni istante
VAR_PLOT = VAR_COLS[0]  # modifica per vedere altre variabili

for pc in range(1, N_PC + 1):
    fig = plots.plot_loading_time(
        P         = model.P,
        col_names = col_names,
        var_name  = VAR_PLOT,
        pc        = pc
    )
    plt.show()

In [ ]:
# Heatmap loadings (variabili aggregate × PC)
# Aggrega i loadings per variabile (media assoluta nel tempo)
V = len(VAR_COLS)
J = N_TIME
P_agg = np.zeros((V, N_PC))
for v in range(V):
    idx = [i for i, c in enumerate(col_names)
           if c.startswith(f'{VAR_COLS[v]}_t')]
    P_agg[v, :] = np.mean(np.abs(model.P[idx, :N_PC]), axis=0)

fig = plots.plot_loading_heatmap(P_agg, VAR_COLS, n_components=N_PC)
plt.title('Heatmap loadings aggregati nel tempo (media |loading|)',
          fontsize=12, fontweight='bold')
plt.show()

## 6 — Diagnostica post-mortem

In [ ]:
# Inizializza monitor post-mortem
pm = BatchPostMortem(
    model        = model,
    n_components = N_PC,
    alpha        = 0.05,
    col_names    = col_names,
    batch_ids    = batch_ids_cal,
    n_time       = N_TIME,
    var_names    = VAR_COLS
)

print(f'Limite T² (95%): {pm.T2_lim:.4f}')
print(f'Limite Q  (95%): {pm.Q_lim:.4f}')

In [ ]:
# Analisi batch di test
result = pm.analyze_batch(X_test_unf, test_batch_ids=batch_ids_test)
print(pm.summary_report(result))

In [ ]:
# Grafico T² e Q per tutti i batch di test
anomaly_idx = result['anomaly_any'].nonzero()[0].tolist()

fig = plots.plot_T2_Q(
    T2            = result['T2'],
    Q             = result['Q'],
    T2_limit      = result['T2_limit'],
    Q_limit       = result['Q_limit'],
    labels        = batch_ids_test,
    highlight_idx = anomaly_idx,
    title         = 'Diagnostica post-mortem — batch di test'
)
plt.show()

In [ ]:
# Score plot test con batch anomali evidenziati
T_test = model.transform(X_test_unf, N_PC)

fig = plots.plot_scores(
    T             = T_test,
    pc_x          = PC_X,
    pc_y          = PC_Y,
    labels        = batch_ids_test,
    highlight_idx = anomaly_idx,
    alpha         = 0.05,
    title         = 'Score plot — batch di test (anomali evidenziati)'
)
plt.show()

In [ ]:
# Contribution plot nel tempo per il primo batch anomalo
if anomaly_idx:
    bi   = anomaly_idx[0]
    bid  = batch_ids_test[bi]
    print(f'Analisi batch anomalo: {bid}')
    detail = result['anomaly_details'].get(bid, {})
    print(f'  Variabili anomale T²: {detail.get("vars_T2", [])}')
    print(f'  Variabili anomale Q : {detail.get("vars_Q",  [])}')

    cq_all, _ = model.contributions_Q(X_test_unf, N_PC)

    for var in detail.get('vars_Q', VAR_COLS[:1]):
        # Limite di controllo per variabile nel tempo
        var_idx  = [i for i, c in enumerate(col_names)
                    if c.startswith(f'{var}_t')]
        ctrl_lim = pm.ctrl_lim_Q[var_idx]

        fig = plots.plot_contribution_time(
            contrib        = cq_all,
            col_names      = col_names,
            var_name       = var,
            batch_idx      = bi,
            stat           = 'Q',
            control_limit  = ctrl_lim
        )
        plt.show()
else:
    print('Nessun batch anomalo rilevato.')

## 7 — Diagnostica on-line (simulazione)

In [ ]:
# Medie di calibrazione per imputazione 'mean'
cal_means = np.nanmean(X_cal_unf, axis=0)

# Inizializza monitor on-line
online = BatchOnlineMonitor(
    model          = model,
    n_components   = N_PC,
    alpha          = 0.05,
    col_names      = col_names,
    n_time         = N_TIME,
    n_vars         = N_VARS,
    var_names      = VAR_COLS,
    impute_method  = 'mean',
    cal_means      = cal_means
)

print('Monitor on-line inizializzato.')
print(f'Metodo imputazione: mean (medie calibrazione)')
print(f'Limite T²: {online.T2_lim:.4f}')
print(f'Limite Q : {online.Q_lim:.4f}')

In [ ]:
# Simula monitoraggio on-line su un batch anomalo
# Scegli quale batch testare
BATCH_TO_MONITOR = 1  # indice nel test set (0-based) — prova 1=thermal, 5=viscosity

x_complete = X_test_unf[BATCH_TO_MONITOR, :]
bid        = batch_ids_test[BATCH_TO_MONITOR]

print(f'Simulazione on-line batch: {bid}')
df_history = online.run_full_batch(x_complete, batch_id=bid)

print(online.summary_report())

# Grafico evoluzione T² e Q nel tempo
fig = online.plot_online_history(df_history)
plt.show()

In [ ]:
# Confronto: batch normale vs batch anomalo
BATCH_NORMAL = 0

online2 = BatchOnlineMonitor(
    model=model, n_components=N_PC, alpha=0.05,
    col_names=col_names, n_time=N_TIME, n_vars=N_VARS,
    impute_method='mean', cal_means=cal_means
)
df_normal = online2.run_full_batch(
    X_test_unf[BATCH_NORMAL, :],
    batch_id=batch_ids_test[BATCH_NORMAL]
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=110)

for ax, df, label, color in [
    (ax1, df_normal,  f'Batch normale ({batch_ids_test[BATCH_NORMAL]})', '#2563EB'),
    (ax2, df_history, f'Batch anomalo ({bid})',                           '#DC2626'),
]:
    ax.plot(df['time'], df['T2'], '-o', color=color,
            linewidth=1.5, markersize=3, label='T²')
    ax.axhline(online.T2_lim, color='red', linestyle='--',
               linewidth=1.2, label=f'Limite T²')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Istante temporale')
    ax.set_ylabel('T²')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Confronto monitoraggio on-line: normale vs anomalo',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8 — AI Interpreter (opzionale)

In [ ]:
from pca_monitor.ai_interpreter import PCAContext, create_interpreter

ctx = PCAContext()
ctx.update(
    process_type      = 'batch',
    section           = 'diagnostics — post-mortem e on-line',
    n_samples         = X_cal_unf.shape[0],
    n_vars            = N_VARS,
    var_names         = VAR_COLS,
    n_components      = N_PC,
    scale_method      = 'auto',
    explained_var     = (model.explained_variance_ratio_[:N_PC]*100).tolist(),
    cumulative_var    = float(model.explained_variance_ratio_[:N_PC].sum()*100),
    rmsecv_values     = rmsecv_vals.tolist(),
    optimal_nc_rmsecv = optimal_nc,
    T2_limit          = pm.T2_lim,
    Q_limit           = pm.Q_lim,
    anomaly_summary   = find_anomalies(
        result['T2'], result['Q'], pm.T2_lim, pm.Q_lim),
    batch_details     = result['anomaly_details'],
    online_summary    = online.summary_report(),
)

ai = create_interpreter(context=ctx, backend='auto')
print('Contesto AI pronto.')

In [ ]:
if ai:
    ai.ask('Quali batch sono risultati anomali e in quale indice '
           'T² o Q? Cosa suggerisce il contribution plot nel tempo?')

In [ ]:
if ai:
    ai.ask('Il monitoraggio on-line ha rilevato la stessa anomalia '
           'del post-mortem? A quale istante temporale è emersa?')